![Platform+](images/banner.png)

<div style="max-width:150px; float:right;">

![The Roastery Coffee Co.](images/roastery-logo.png)

</div>

# Lab 10: Capstone — Roastery Intelligence
## Feed the platform: build the pipeline everything else depends on.

> **Persona track: Data Engineer** — same mission, your angle.

:point_right: **In this lesson, you'll build a production-style pipeline that keeps enriched Roastery data fresh — and keep it running even when something breaks.**

The scenario: The VP of Operations flagged a revenue dip at Roastery Mission, and the team's analysis of it depends on current, trustworthy data. Your job is to build and operate the pipeline that feeds that investigation — not just once, but on an ongoing basis.

We'll cover:
1. Investigate — is the source trustworthy? (join integrity + completeness)
2. Consume a governed handoff — look closely before you build on it
3. Build — ingest + enrich with AI
4. Handle it when something goes wrong mid-build (Time Travel recovery)
5. Structure the unstructured — parse PDFs into a queryable table
6. Share the platform — grants + marketplace
7. Automate — keep it fresh (Dynamic Table + Task)
8. Verify your work and reflect on what could still go wrong

📚 **By the end you'll be able to:**
- Turn a pipeline goal into a self-authored CoCo prompt, not a copy-pasted one
- Recognise when a handed-off dataset isn't as safe as it looks
- Recover from a broken pipeline step without losing your work
- Automate access and scheduling so the pipeline runs without you

**Time:** ~45 min · **Lab interface:** Cortex Code (CoCo) is your tutor — ask it to explain or fix anything.
*Source data lives in the shared, read-only `PLATPLUS_DB`; anything you create lives in your own database (named after your username).*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
my_analyst_role = user + '_ANALYST_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)


## Set your session context
Every query runs under a role (what you're allowed to do) and a warehouse (compute). The cell below switches to your own role, creates and uses your own warehouse, and creates a `RAW` schema for what you'll build in this lab.

In [ ]:
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.RAW;


> :mortar_board: **This is a capstone — not a tutorial.** Every section calls back to a skill from an earlier lab. Some cells are pre-filled (run them); others are yours to write. Each section gives you **three paths to the answer (your choice)** — you can try all three if you wish, but you only need one:
>
> 1. **Write it yourself** in the empty cells (the real challenge — prove you learned it)
> 2. **Reveal the 🧩 Stuck? Reference solution** and paste it in (see the answer directly)
> 3. **Use the 🤖 CoCo assist prompt** to have CoCo generate the code for you (practice the AI-assisted workflow)
>
> All three paths produce the same result going into the same empty cells. Pick whichever level of help you need — there's no penalty for using the hints.

> :rotating_light: **IMPORTANT — CoCo vs notebook SQL cells:** As you go through this lab and use CoCo, be aware that curly braces `{}` in SQL (e.g. `{'mode': 'LAYOUT'}`) can conflict with CoCo's internal SQL execution. If CoCo reports an error but the SQL looks correct, just copy it into a notebook SQL cell and run it there — it will work fine. This is a difference in how CoCo executes SQL internally vs how notebook cells send it directly to Snowflake.

---
## 1 · Investigate — is the source trustworthy? (Labs 01, 02 — Navigate + Verify)

The DA's dip analysis and the PA's audit both assumed the underlying data was solid. Your job: check that assumption before building anything on top of it.

**Why this matters (data engineer perspective):** correctness isn't just "does the query run" — it's "does every row have a home." An orphaned foreign key silently drops rows out of a JOIN with no error, which is worse than a query that fails loudly.

In [ ]:
-- Does every FEEDBACK row have a matching STORES row? (silent data-quality check)
SELECT COUNT(*) AS orphan_feedback_rows
FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK f
LEFT JOIN PLATPLUS_DB.ROASTERY_SALES.STORES s ON s.store_id = f.store_id
WHERE s.store_id IS NULL;


> 🔎 **Review.** Zero orphan rows means every feedback record joins cleanly to a real store — the data feeding the investigation is joinable and current. If this ever returns > 0, that's a data-quality incident to raise before anyone builds a dashboard on top of it.

### Before you build — is the source complete and unique?
A join-integrity check proves rows *join*. It doesn't prove the source is *complete*. Lab 02's habit: verify row counts, key uniqueness, and NULLs on the input **before** you build a pipeline on top of it.

> 📌 **Recall Lab 02:** verify results — don't assume a handoff is clean just because it loads.

In [ ]:
-- Lab 02: completeness + uniqueness + NULL check on the handoff, before building on it
SELECT
  COUNT(*)                               AS total_rows,
  COUNT(DISTINCT feedback_id)            AS distinct_keys,
  COUNT(*) - COUNT(DISTINCT feedback_id) AS duplicate_keys,
  COUNT(*) - COUNT(comment)              AS null_comments,
  COUNT(*) - COUNT(store_id)             AS null_store_ids
FROM PLATPLUS_DB.ROASTERY_INTELLIGENCE.V_FEEDBACK_SAFE;


> 🔎 **Review.** Expect **269 rows, 269 distinct keys, 0 duplicates, 0 NULL comments, 0 NULL store IDs.** The source is complete, uniquely keyed, and fully populated — safe to build a pipeline on. If `DUPLICATE_KEYS > 0`, a downstream JOIN would fan out and inflate every count silently; if `NULL_COMMENTS > 0`, the enrichment step would skip those rows. You verify this *once, up front* — not after a dashboard looks wrong.

---
## 2 · Consume 📦 — a governed handoff (Labs 03, 05 — RBAC + AI)

In a real workplace, a Platform Admin team would publish governed views for engineers to consume. In this lab, that's already been done for you — `PLATPLUS_DB.ROASTERY_INTELLIGENCE.V_FEEDBACK_SAFE` is a pre-provisioned view that your role (`{{my_role}}`) has SELECT on.

**What the view does behind the scenes:** It internally joins FEEDBACK to CUSTOMERS (using the view *owner's* privileges, not yours) but only exposes the safe columns — feedback text, scores, and store context. No customer names, no customer IDs. You can't reach `CUSTOMERS` directly; this view is your only controlled path to that data, and it decides what you see.

> 📌 **You don't need to run any other lab first.** This view is already provisioned in the shared `PLATPLUS_DB` database.

Run the cell below to see what the view gives you.

In [ ]:
SELECT * FROM PLATPLUS_DB.ROASTERY_INTELLIGENCE.V_FEEDBACK_SAFE
LIMIT 10;


> 🔎 **Review.** No `FULL_NAME`, no `customer_id` — just feedback joined to store context. The view intentionally excludes customer PII. But what if you tried to get those names yourself by joining FEEDBACK to CUSTOMERS directly? Your role (`{{my_role}}`) has SELECT on the view — let's see if it has access to the underlying tables...

In [ ]:
-- The "natural shortcut" — join FEEDBACK straight to CUSTOMERS yourself instead of using the published view
SELECT f.feedback_id, c.full_name, f.comment
FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK f
JOIN PLATPLUS_DB.ROASTERY_SALES.CUSTOMERS c ON c.customer_id = f.customer_id
LIMIT 5;


> 🔒 **Access denied — and that's the point.** Your role (`{{my_role}}`) has SELECT on the **view** but NOT on the raw `CUSTOMERS` table. This is how views enforce governance — the view runs with its *owner's* privileges (not yours), so it can read CUSTOMERS internally while you cannot. You already saw the view work two cells above — that's your proof the governed path gives you what you need. The direct path is blocked; the view is the answer. Build on `V_FEEDBACK_SAFE`, not a hand-rolled join.

---
## 3 · Build — ingest + enrich (Labs 04, 05 — Cost + AI)

Copy the governed handoff into your own database (writable, trackable), then enrich it.

**Step 1 — a local, writable copy.**

> 📌 **Why copy?** `PLATPLUS_DB` is shared and read-only — you can SELECT from it but you can't write to it, add columns, or run AI enrichment in place. To build a pipeline that transforms the data, you need a writable copy in your own database.

In [ ]:
CREATE OR REPLACE TABLE {{my_db}}.RAW.FEEDBACK_SOURCE AS
SELECT * FROM PLATPLUS_DB.ROASTERY_INTELLIGENCE.V_FEEDBACK_SAFE;


**Estimate before you scale.** You're about to run `AI_COMPLETE` on every comment. Lab 04's habit: price the job *before* you run it. `AI_COUNT_TOKENS` returns the token count a prompt would consume — multiply by your row count to size the spend before committing compute.

> 📌 **Recall Lab 04:** cost before compute — a token estimate is cheaper than a surprise bill.

In [ ]:
-- Lab 04: estimate the enrichment cost BEFORE running it at scale
SELECT
  COUNT(*) AS rows_to_enrich,
  SUM(AI_COUNT_TOKENS('ai_complete', 'llama3.1-8b',
      'Rewrite this customer feedback comment in exactly one short sentence, '
      || 'summarizing the main point neutrally. Comment: ' || comment)) AS total_prompt_tokens,
  ROUND(AVG(AI_COUNT_TOKENS('ai_complete', 'llama3.1-8b',
      'Rewrite this customer feedback comment in exactly one short sentence, '
      || 'summarizing the main point neutrally. Comment: ' || comment)), 1) AS avg_tokens_per_row
FROM {{my_db}}.RAW.FEEDBACK_SOURCE
WHERE comment IS NOT NULL;


> 🔎 **Review.** Roughly **269 rows, ~11,500 prompt tokens total, ~43 tokens/row.** Small here — but this is the exact reflex that matters when the table is 269 *million* rows instead of 269. You size the job first, then decide whether to run it, batch it, or pick a cheaper model. (This counts *prompt* tokens only; output tokens add to the total but scale the same way.)

> ⚠️ Keep the enrich prompt string **identical** to the one in `resolution_enrichment` so the estimate matches what actually runs.

**Step 2 — enrich it with AI_COMPLETE.**

> 🤖 **CoCo assist** *(try the empty cell yourself first — if stuck, reveal the 🧩 solution below OR ask CoCo in your own words):*
>
> Describe what you want CoCo to build. Your prompt should describe:
> - A table `{{my_db}}.RAW.FEEDBACK_ENRICHED` built from `{{my_db}}.RAW.FEEDBACK_SOURCE`
> - A new column named exactly `standardized_description` — use `AI_COMPLETE` to rewrite each `comment` into one short, neutral summary sentence
> - Only rows where `comment IS NOT NULL`
>
> Paste CoCo's SQL into the cell below and run it. Iterate if it errors.

In [ ]:
-- YOUR TURN: paste CoCo's enrichment SQL on the blank line below, then run it


**Paste into the `build_enrich_paste_cell` cell (the YOUR TURN cell above):**

<details><summary>🧩 Stuck? Reference solution</summary>

```sql
CREATE OR REPLACE TABLE {{my_db}}.RAW.FEEDBACK_ENRICHED AS
SELECT feedback_id, store_id, store_name, feedback_date, score, comment,
       AI_COMPLETE(
         'llama3.1-8b',
         'Rewrite this customer feedback comment in exactly one short sentence, ' ||
         'summarizing the main point neutrally. Comment: ' || comment
       ) AS standardized_description
FROM {{my_db}}.RAW.FEEDBACK_SOURCE
WHERE comment IS NOT NULL;
```

The key: **`AI_COMPLETE`** with a short, neutral-rewrite prompt applied to each comment. The model choice (**`llama3.1-8b`**) balances cost and quality for a high-volume rewrite.
</details>

In [ ]:
SELECT COUNT(*) AS total, COUNT(standardized_description) AS enriched
FROM {{my_db}}.RAW.FEEDBACK_ENRICHED;


> 🔎 **Review.** `TOTAL` and `ENRICHED` should match — every row with a comment got a `standardized_description`. This is a one-time CTAS, a snapshot. It's not connected to anything yet, and it's not automatically kept current. Before you wire it into anything permanent, though — an incident just landed on your desk.

---
## 4 · Oops — Recovery (Lab 08 — Time Travel)

### An incident, not an exercise

The next cell is **pre-written, not something you author** — it simulates something that already happened: last night's automated run had a bug. This mirrors Lab 8's exact recovery mechanism, just applied to your own pipeline instead of a practice table.

In [ ]:
-- SIMULATED INCIDENT (pre-written — this is not something you'd write yourself)
-- Last night's automated run had a bug: it nulled out one store's enrichment.
UPDATE {{my_db}}.RAW.FEEDBACK_ENRICHED
SET standardized_description = NULL
WHERE store_name = 'Roastery Mission';
SET bad_update = LAST_QUERY_ID();

-- Capture the query ID of the bad UPDATE so Time Travel can reference it later.
-- In the real world you wouldn't have LAST_QUERY_ID() handy — you'd interrogate
-- QUERY_HISTORY to find the offending statement's query ID.

SELECT COUNT(*) AS total, COUNT(standardized_description) AS still_enriched
FROM {{my_db}}.RAW.FEEDBACK_ENRICHED;


> 🔎 **Review.** `STILL_ENRICHED` dropped below `TOTAL` — some rows lost their `standardized_description` overnight. This is the kind of thing that surfaces as "why does the dashboard look wrong" the next morning, not as an error anyone saw happen.

In [ ]:
-- Confirm the pre-incident state is still recoverable (Lab 8's BEFORE pattern)
SELECT COUNT(standardized_description) AS enriched_before_bug
FROM {{my_db}}.RAW.FEEDBACK_ENRICHED BEFORE(statement => $bad_update);


> ✅ **Nothing was truly lost.** `BEFORE(statement => $bad_update)` sees the table exactly as it was one statement earlier — full enrichment intact. Recover it properly rather than just re-running the AI function (that would burn tokens re-processing rows that were never touched).

In [ ]:
-- Clone the good state
CREATE OR REPLACE TABLE {{my_db}}.RAW.FEEDBACK_ENRICHED_RECOVERED
CLONE {{my_db}}.RAW.FEEDBACK_ENRICHED BEFORE(statement => $bad_update);

SELECT COUNT(*) AS total, COUNT(standardized_description) AS enriched
FROM {{my_db}}.RAW.FEEDBACK_ENRICHED_RECOVERED;


In [ ]:
-- Verified good — swap it in with a two-statement DROP + RENAME (note the brief window where the table doesn't exist; the production-safe ALTER TABLE ... SWAP WITH alternative is in the teaching note below)
DROP TABLE {{my_db}}.RAW.FEEDBACK_ENRICHED;
ALTER TABLE {{my_db}}.RAW.FEEDBACK_ENRICHED_RECOVERED RENAME TO {{my_db}}.RAW.FEEDBACK_ENRICHED;


In [ ]:
SELECT COUNT(*) AS total, COUNT(standardized_description) AS enriched,
       COUNT(*) - COUNT(standardized_description) AS still_null
FROM {{my_db}}.RAW.FEEDBACK_ENRICHED;


> ✅ **`STILL_NULL` should read 0.** Full enrichment restored, without re-running a single AI call. Only now — after proving the pipeline is trustworthy — do you share it and automate it.

> **What just happened:** We dropped the corrupted table and renamed the cloned (pre-incident) copy into its place — a two-statement "swap-in." It works, but there's a brief moment between the `DROP` and the `ALTER ... RENAME` where the table doesn't exist at all. Any query that hits the table in that window gets an error.
>
> **Production alternative — `ALTER TABLE ... SWAP WITH`:**
> ```sql
> ALTER TABLE FEEDBACK_ENRICHED SWAP WITH FEEDBACK_ENRICHED_RECOVERED;
> DROP TABLE FEEDBACK_ENRICHED_RECOVERED;  -- now holds the bad data
> ```
> `SWAP WITH` exchanges the two tables' contents atomically in a single metadata operation — no window of unavailability. Prefer it when the table is actively queried by dashboards or downstream pipelines.

---
## 5 · Structure the unstructured — coffee knowledge base (Lab 09 — Unstructured AI)

The platform doesn't only serve rows — it serves documents too. The Roastery has a folder of coffee spec sheets (PDFs) on a stage. Turn them into a **structured, queryable table**: one row per drink, full text extracted. This is what "getting unstructured data into the platform" looks like as a pipeline, not a one-off.

> 📌 **Recall Lab 09:** `AI_PARSE_DOCUMENT` turns a PDF on a stage into text you can store and query.

In [ ]:
-- What documents are on the stage? (DIRECTORY lists staged files as rows — a pipeline over files)
SELECT relative_path, size
FROM DIRECTORY(@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE)
WHERE relative_path LIKE 'coffee_kb/%.pdf'
ORDER BY relative_path;


**Your task:** build `{{my_db}}.RAW.COFFEE_KB` — one row per PDF in `coffee_kb/`, with the drink name and the extracted document text. The pattern: drive off `DIRECTORY()` so it scales to any number of files, and use `AI_PARSE_DOCUMENT(... {'mode':'LAYOUT'}):content::STRING` to extract each file's text.

> 🤖 **CoCo assist** *(try the empty cell yourself first — if stuck, reveal the 🧩 solution below OR use this prompt):*
>
> *"Write a CREATE TABLE AS SELECT that reads every PDF under coffee_kb/ on a stage using DIRECTORY(), and for each one extracts the text with AI_PARSE_DOCUMENT in LAYOUT mode into a column called doc_text, plus a drink column from the filename."*

In [ ]:
-- YOUR TURN: paste CoCo's COFFEE_KB build on the blank line below, then run it


**Paste into the `build_coffee_kb` cell (the YOUR TURN cell above):**

<details><summary>🧩 Stuck? Reference solution</summary>

```sql
CREATE OR REPLACE TABLE {{my_db}}.RAW.COFFEE_KB AS
SELECT
  REGEXP_REPLACE(relative_path, '^coffee_kb/|\\.pdf$', '') AS drink,
  relative_path AS source_file,
  AI_PARSE_DOCUMENT(
    TO_FILE('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE', relative_path),
    {'mode': 'LAYOUT'}
  ):content::STRING AS doc_text
FROM DIRECTORY(@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE)
WHERE relative_path LIKE 'coffee_kb/%.pdf';
```

Driving off `DIRECTORY()` (not a hardcoded filename list) is the pipeline instinct: the same statement handles 8 files or 800. `AI_PARSE_DOCUMENT` in `LAYOUT` mode returns a JSON object; `:content::STRING` pulls out the plain text. `REGEXP_REPLACE` strips the folder prefix and `.pdf` suffix to get a clean `drink` key.
</details>

In [ ]:
SELECT drink, source_file, LENGTH(doc_text) AS text_len
FROM {{my_db}}.RAW.COFFEE_KB
ORDER BY drink;


> 🔎 **Review.** Expect **8 rows** — americano, cappuccino, cold_brew, espresso, flat_white, latte, mocha, pour_over — each with ~1,800–2,200 characters of extracted text. Unstructured PDFs are now structured rows an analyst can query, join, or run more AI functions over. That's the whole job: make the unqueryable queryable.

---
## 6 · Share the platform — grants + marketplace (Labs 03, 06 — RBAC + Marketplace)

### Share it (future grants)

Now that the pipeline is trustworthy, downstream analysts should get access to whatever it produces from now on — automatically, not via a ticket.

To reach a table, a role needs three layers of access:
1. **`USAGE` on the database** — permission to "enter" the database
2. **`USAGE` on the schema** — permission to see objects inside
3. **`SELECT` on the table/view** — permission to read rows

The stored procedure `APPLY_FUTURE_GRANTS` handles layer 3 (SELECT on all existing + future tables). But layers 1 and 2 must be granted separately — the procedure doesn't do them. We grant all three here so this notebook is self-contained, regardless of whether you ran the platform-admin track first.

In [ ]:
USE ROLE {{my_role}};
USE WAREHOUSE {{my_wh}};

-- Layers 1 & 2: let the analyst role enter the database and schema
GRANT USAGE ON DATABASE {{my_db}}     TO ROLE {{my_analyst_role}};
GRANT USAGE ON SCHEMA {{my_db}}.RAW   TO ROLE {{my_analyst_role}};

-- Layer 3: SELECT on all existing + future tables in this schema
CALL PLATPLUS_DB.PUBLIC.APPLY_FUTURE_GRANTS('{{my_db}}.RAW', '{{my_analyst_role}}');


In [ ]:
-- Can {{my_analyst_role}} see the enriched table WITHOUT a new grant?
USE ROLE {{my_analyst_role}};
USE WAREHOUSE PLATPLUS_WH;  -- (Uses PLATPLUS_WH because the analyst role doesn't have USAGE on your personal warehouse)
SELECT COUNT(*) FROM {{my_db}}.RAW.FEEDBACK_ENRICHED;


> ✅ **It works — no additional grant needed.** Anything your pipeline creates in `{{my_db}}.RAW` from now on, `{{my_analyst_role}}` can already see. That's the point of setting this up once, at the schema level.
>
> The USAGE grants are idempotent — if the platform-admin track already ran them, re-running here is a harmless no-op.

### Make the marketplace data consumable
The analyst wants the Alpine Peaks book catalog (a Marketplace listing) for the book–coffee pairing work — but they shouldn't each wrangle the raw shared database. Engineer's job: publish a clean, curated view over the marketplace data and grant the analyst access to *that*.

> 📌 **Recall Lab 06:** a Marketplace listing is just another database you can build on — shape it into something your consumers can use.

> 📌 **The `ALPINE_PEAKS_PUBLISHING` database is already here** — it was provisioned as part of the account setup. The cell below calls `ACCESS_ALPINE_PEAKS_PUBLISHING_DATA` to ensure your role has `IMPORTED PRIVILEGES` on the shared database (Lab 06 would have done this, but we include it here so the capstone is self-contained).

In [ ]:
USE ROLE {{my_role}};
USE WAREHOUSE {{my_wh}};

-- Ensure our role can access the shared marketplace database
-- (Lab 06 would have done this; we repeat it here so the capstone is self-contained)
CALL PLATPLUS_DB.RESOURCES.ACCESS_ALPINE_PEAKS_PUBLISHING_DATA('{{my_role}}');

In [ ]:
USE ROLE {{my_role}};
USE WAREHOUSE {{my_wh}};

CREATE SCHEMA IF NOT EXISTS {{my_db}}.SANDBOX;

-- Curate the marketplace listing into a consumable view, then grant the analyst access
CREATE OR REPLACE VIEW {{my_db}}.SANDBOX.V_BOOK_CATALOG AS
SELECT title, author, category, release_date
FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES
ORDER BY release_date DESC;

GRANT USAGE ON DATABASE {{my_db}}          TO ROLE {{my_analyst_role}};
GRANT USAGE ON SCHEMA {{my_db}}.SANDBOX    TO ROLE {{my_analyst_role}};
GRANT SELECT ON VIEW {{my_db}}.SANDBOX.V_BOOK_CATALOG TO ROLE {{my_analyst_role}};


In [ ]:
-- Confirm the analyst role can now consume the curated catalog
USE ROLE {{my_analyst_role}};
USE WAREHOUSE PLATPLUS_WH;  -- (PLATPLUS_WH is a shared warehouse the analyst role has access to)
SELECT COUNT(*) AS books_visible, COUNT(DISTINCT category) AS categories
FROM {{my_db}}.SANDBOX.V_BOOK_CATALOG;


> ✅ **The analyst sees 88 books across 8 categories through your view** — without ever touching the raw shared database. You decided the shape (columns, ordering); they consume the result. That's the engineer's role in a Marketplace workflow: turn a raw listing into a governed, purpose-built product.

---
## 7 · Automate — keep it fresh (Lab 07 — Tasks/Loading)

Two moving parts, each doing one job:
- A **Dynamic Table** re-runs the enrichment automatically whenever `FEEDBACK_SOURCE` changes — no scheduling code, incremental by default.
- A **Task** keeps `FEEDBACK_SOURCE` itself in sync with the upstream handoff, on a schedule.

**Step 1 — promote the enrichment to a Dynamic Table.**

In [ ]:
-- Return to your own role before continuing
USE ROLE {{my_role}};
USE WAREHOUSE {{my_wh}};

CREATE OR REPLACE DYNAMIC TABLE {{my_db}}.RAW.FEEDBACK_ENRICHED_LIVE
  TARGET_LAG = '1 hour'
  WAREHOUSE  = PLATPLUS_WH
AS
SELECT feedback_id, store_id, store_name, feedback_date, score, comment,
       AI_COMPLETE(
         'llama3.1-8b',
         'Rewrite this customer feedback comment in exactly one short sentence, ' ||
         'summarizing the main point neutrally. Comment: ' || comment
       ) AS standardized_description
FROM {{my_db}}.RAW.FEEDBACK_SOURCE
WHERE comment IS NOT NULL;


In [ ]:
SELECT COUNT(*) AS total, COUNT(standardized_description) AS enriched
FROM {{my_db}}.RAW.FEEDBACK_ENRICHED_LIVE;


> 🔎 **Review.** Same enrichment logic as your recovered table — but now it's a Dynamic Table, refreshing automatically within `TARGET_LAG` whenever `FEEDBACK_SOURCE` changes. You develop with CTAS, validate it, then promote to a Dynamic Table once it's proven — the exact pattern from Lab 5.

**Step 2 — a Task to keep `FEEDBACK_SOURCE` current.** The Dynamic Table only re-enriches when its source changes — something still has to bring in new rows from the upstream handoff. That's this Task's one job.

In [ ]:
CREATE OR REPLACE TASK {{my_db}}.RAW.SYNC_FEEDBACK_SOURCE_TASK
  WAREHOUSE = {{my_wh}}
  SCHEDULE  = '5 MINUTE'
AS
MERGE INTO {{my_db}}.RAW.FEEDBACK_SOURCE tgt
USING PLATPLUS_DB.ROASTERY_INTELLIGENCE.V_FEEDBACK_SAFE src
ON tgt.feedback_id = src.feedback_id
WHEN NOT MATCHED THEN
  INSERT (feedback_id, store_id, store_name, feedback_date, score, comment)
  VALUES (src.feedback_id, src.store_id, src.store_name, src.feedback_date, src.score, src.comment);

-- New tasks start suspended by default — confirmed below, then started explicitly
SHOW TASKS LIKE 'SYNC_FEEDBACK_SOURCE_TASK' IN SCHEMA {{my_db}}.RAW;


**Step 3 — actually run it.** Every other Task in this course was created then immediately suspended (Lab 7's "just a taster"). This time, prove it works end-to-end: resume it, fire it once on demand, check the history, then suspend it again.

In [ ]:
ALTER TASK {{my_db}}.RAW.SYNC_FEEDBACK_SOURCE_TASK RESUME;
EXECUTE TASK {{my_db}}.RAW.SYNC_FEEDBACK_SOURCE_TASK;


> ⏱️ **Run the next cell** — it pauses for 30 seconds while the Task executes, then confirms completion. (The cell shows results-only; just select it and press Shift+Enter to run.)

In [ ]:
import time
time.sleep(30)
print('🀀' * 30 + '\n\nDone — 30 seconds elapsed. Run the next cell to check TASK_HISTORY.')

In [ ]:
SELECT name, state, scheduled_time, completed_time, error_message
FROM TABLE({{my_db}}.INFORMATION_SCHEMA.TASK_HISTORY(
  TASK_NAME => 'SYNC_FEEDBACK_SOURCE_TASK',
  SCHEDULED_TIME_RANGE_START => DATEADD(minute, -10, CURRENT_TIMESTAMP())
))
ORDER BY scheduled_time DESC
LIMIT 5;


In [ ]:
-- IMPORTANT: suspend the Task so it doesn't keep running on its 5-minute schedule
ALTER TASK {{my_db}}.RAW.SYNC_FEEDBACK_SOURCE_TASK SUSPEND;


> 🔎 **Review.** `STATE` should read `SUCCEEDED`. You proved the full loop: resume → run → verify → suspend — the same lifecycle a production Task goes through on every deploy, just compressed into one lab.

---
## 8 · Verify + wrap

In [ ]:
-- Structural check: does the final pipeline output carry any customer PII column at all?
DESCRIBE TABLE {{my_db}}.RAW.FEEDBACK_ENRICHED_LIVE;


> ✅ **No `full_name`, no `customer_id` — none of the columns are customer identifiers.** Not because you remembered to filter one out, but because the only path into this pipeline (`V_FEEDBACK_SAFE`) never had PII in the first place, and the raw shortcut was access-denied. The governance held structurally, not just by convention.

### 🧭 Open reflection

Before you wrap up, answer these for yourself (or write them down — there's no single right answer):

- **What could still go wrong?** You've handled one incident (the corruption twist). What's a *different* failure mode this pipeline hasn't been tested against?
- **The access-denied trap** — if `CUSTOMERS` access hadn't been revoked, would you have caught the PII risk yourself? What does that tell you about relying on discipline vs. relying on enforcement?
- **What's still missing?** A Dynamic Table and a Task cover the happy path. What would you add before calling this production-ready — monitoring, alerting, a dead-letter pattern for failed enrichments?
- **Which lab skill was hardest to recall?** That's your signal for what to review before building pipelines for real.
- **CoCo usage:** where did you ask CoCo to write SQL for you (enrich, COFFEE_KB)? Knowing *what to ask* is the production skill — not memorizing syntax.

### Check your work
Run the next cell to **check your work**. It confirms the pipeline is intact, recovered, and shared — no need to read the code.

> 📌 **Note:** This only checks sections that create persistent objects (tables, views, grants). Sections that produce one-off query results (join-integrity checks, access-denied demos, token estimates) aren't checked here — you verified those by reading their output as you went.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'

try:
    row = session.sql(f"""
        SELECT COUNT(*) AS total, COUNT(standardized_description) AS enriched
        FROM {my_db}.RAW.FEEDBACK_ENRICHED_LIVE
    """).collect()[0]
    total, enriched = row[0], row[1]
    enrich_ok = total >= 1 and total == enriched
except Exception:
    total, enriched, enrich_ok = None, None, False

try:
    cols = [c[0].upper() for c in session.sql(f"DESCRIBE TABLE {my_db}.RAW.FEEDBACK_ENRICHED_LIVE").collect()]
    pii_ok = 'FULL_NAME' not in cols and 'CUSTOMER_ID' not in cols
except Exception:
    pii_ok = False

try:
    tasks = session.sql(f"SHOW TASKS LIKE 'SYNC_FEEDBACK_SOURCE_TASK' IN SCHEMA {my_db}.RAW").collect()
    task_ok = len(tasks) >= 1
except Exception:
    task_ok = False

try:
    coffee_kb_row = session.sql(f"SELECT COUNT(*) FROM {my_db}.RAW.COFFEE_KB").collect()[0]
    coffee_kb_ok = coffee_kb_row[0] >= 1
    coffee_kb_count = coffee_kb_row[0]
except Exception:
    coffee_kb_ok = False
    coffee_kb_count = 0

try:
    book_catalog_row = session.sql(f"SELECT COUNT(*) FROM {my_db}.SANDBOX.V_BOOK_CATALOG").collect()[0]
    book_catalog_ok = book_catalog_row[0] >= 1
    book_catalog_count = book_catalog_row[0]
except Exception:
    book_catalog_ok = False
    book_catalog_count = 0

print(f"{'✅' if enrich_ok else '⛔'}  FEEDBACK_ENRICHED_LIVE fully enriched, no NULLs  {'(' + str(enriched) + '/' + str(total) + ')' if total is not None else '(not found)'}")
print(f"{'✅' if pii_ok else '⛔'}  No customer PII columns in the final pipeline output")
print(f"{'✅' if task_ok else '⛔'}  SYNC_FEEDBACK_SOURCE_TASK exists")
print(f"{'✅' if coffee_kb_ok else '⛔'}  COFFEE_KB parsed  ({'expect 8, got ' + str(coffee_kb_count) if coffee_kb_count else 'not found'})")
print(f"{'✅' if book_catalog_ok else '⛔'}  V_BOOK_CATALOG consumable  ({'expect 88, got ' + str(book_catalog_count) if book_catalog_count else 'not found'})")
print('-' * 52)
if enrich_ok and pii_ok and task_ok and coffee_kb_ok and book_catalog_ok:
    print('🏁  Roastery Intelligence — Data platform layer — COMPLETE')
else:
    print('⚠️  Not all checks passed yet — re-run the relevant cell above, then run this cell again.')

### Labs recalled in this capstone

| Lab | Skill | Where it appeared |
|-----|-------|-------------------|
| 01 | Navigate objects & joins | Section 1 — join-integrity check |
| 02 | Verify results | Section 1 — completeness + uniqueness check |
| 03 | RBAC & access | Section 2 — access-denied shortcut trap; Section 6 — future grants |
| 04 | Credits & cost | Section 3 — AI_COUNT_TOKENS estimate |
| 05 | AI functions | Section 3 — AI_COMPLETE enrichment |
| 06 | Marketplace | Section 6 — curated V_BOOK_CATALOG for the analyst |
| 07 | Getting data in / Tasks | Section 7 — Dynamic Table + sync Task |
| 08 | Time Travel | Section 4 — corrupt → BEFORE → clone → swap recovery |
| 09 | Unstructured AI | Section 5 — parse 8 PDFs into COFFEE_KB |

---
### ❄️ Key Takeaways
- A join-integrity check (orphan rows) catches silent data-quality problems that a working query would never surface
- An access-denied error on a "natural shortcut" isn't a bug to work around — it's the governance layer doing its job
- Time Travel's `BEFORE(statement => ...)` + clone + swap-in recovers a corrupted table without re-running expensive AI calls on untouched rows
- Prove a pipeline is trustworthy *before* sharing it (future grants) or automating it (a Task) — not after
- A Dynamic Table and a Task can divide responsibility cleanly: one keeps enrichment current, the other keeps the source current
- Governance that depends only on discipline is weaker than governance enforced structurally — the difference showed up directly in this lab's own pipeline
- Estimate cost with `AI_COUNT_TOKENS` before running an AI function at scale — the reflex matters more at 269M rows than at 269
- `DIRECTORY()` turns a stage full of files into rows — the foundation of any document pipeline
- Curating a Marketplace listing into a governed view is how an engineer turns raw shared data into a consumable product

**The Other Personas:**
- The **Data Analyst** notebook consumes a different handoff (`V_PRODUCTS_ENRICHED`) and builds the reusable view this pipeline's data ultimately feeds
- The **Platform Admin** notebook governs a PII leak in a summary table and audits the cost/access of the whole investigation

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.